# A2 RNN's full model

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [26]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell, Input, Concatenate, Lambda, Attention
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

### Stuff

In [2]:
from scipy.ndimage import rotate
tf.random.set_seed(42)

# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


### More stuff

In [ ]:
# Scheduled masking
class ScheduledSamplingCallback(tf.keras.callbacks.Callback):
    def __init__(self, prob_var, decay_rate=0.05, min_prob=0.1):
        super().__init__()
        self.prob_var = prob_var
        self.decay_rate = decay_rate
        self.min_prob = min_prob

    def on_epoch_end(self, epoch, logs=None):
        # Calculate the new value
        current_val = self.prob_var.numpy()
        new_val = max(self.min_prob, current_val - self.decay_rate)
        
        # Update the TensorFlow variable
        tf.keras.backend.set_value(self.prob_var, new_val)
        
        # Log to console for monitoring
        print(f"\n --- End of Epoch {epoch + 1}: Teacher Forcing Ratio set to {new_val:.2f} ---")


# 1. Initialize the Global Variable for Scheduled Sampling
# This must exist in the global scope so the Callback and Lambda can see it.
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32, name="sampling_prob_global")

import tensorflow as tf
import numpy as np

# Initialize the probability variable globally
# start at 1.0 (100% Teacher Forcing)
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32, name="global_sampling_prob")

def robust_scheduled_mask(x, training=None):
    """
    Academically robust wrapper for scheduled sampling.
    The 'training' argument is required to satisfy the Keras Functional API signature.
    """
    # We ignore the 'training' flag here because our sampling_prob variable 
    # handles the logic, but the argument MUST be present in the signature.
    return scheduled_mask_layer(x, training=training, prob=sampling_prob)

In [37]:
# Initial probability variable (start at 100% teacher forcing)
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32)

def train_warmup(model, X_img, Y_ans_in, Y_ans_target, val_data, epochs=5):
    # 1. Structural Constraint: Freeze the Encoder
    grafted_encoder.trainable = False
    
    # 2. Compile: Setting the Optimization Manifold
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    # 3. Callback: Correctly referencing the global sampling_prob
    sampling_callback = ScheduledSamplingCallback(
        prob_var=sampling_prob, 
        decay_rate=0.05, 
        min_prob=0.7
    )
    
    # 4. Rank-5 Enforcement for ConvLSTM2D
    if len(X_img.shape) == 4:
        X_img = X_img[..., tf.newaxis]
    
    X_val, Y_val_in = val_data[0]
    Y_val_target = val_data[1]
    if len(X_val.shape) == 4:
        X_val = X_val[..., tf.newaxis]

    print("Starting Warm-up Training...")
    # Explicitly mapping X and Y lists to avoid positional ValueError
    history = model.fit(
        x=[X_img, Y_ans_in], 
        y=Y_ans_target,
        validation_data=([X_val, Y_val_in], Y_val_target),
        epochs=epochs,
        callbacks=[sampling_callback]
    )
    return history

def train_fine_tune(model, train_data, val_data, epochs=10):
    # 1. Unfreeze the Encoder
    grafted_encoder.trainable = True
    
    # 2. Compile with lower Learning Rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    # 3. Callback (pushing to autonomy)
    sampling_callback = ScheduledSamplingCallback(sampling_prob, decay_rate=0.1, min_prob=0.1)
    
    # 4. Fit with explicit x and y
    print("Starting Fine-tuning...")
    model.fit(
        x=train_data[0], 
        y=train_data[1],
        validation_data=(val_data[0], val_data[1]),
        epochs=epochs,
        callbacks=[sampling_callback]
    )
    return model

In [8]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [9]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [10]:
# Full pipeline training data
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

## Full model

In [15]:
# Load the visual encoder and calculator, which were saved in A2_RNNs_Joep_preprocessing.ipynb.
from tensorflow.keras.saving import load_model
visual_encoder = load_model('visual_encoder.keras',safe_mode=False)
calculator = load_model('calculator.keras', safe_mode=False)

#visual_encoder.summary()
calculator.summary()

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 4, 15)     │          0 │ answer[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ scheduled_maskin… │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,089,775 (7.97 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,393,184 (5.31 MB)

In [16]:
from tensorflow.keras.models import Model

# 1. Use the EXACT symbolic inputs that already exist in the model
# This avoids the "empty inputs" and "NotImplemented" errors
encoder_inputs = visual_encoder.inputs 

# 2. Reference the internal layer outputs directly
# Output 0: The final softmax predictions (Symbolic Head)
predictions = visual_encoder.get_layer('output_visual_encoder').output

# Output 1: The full sequence of hidden states from the LSTM (Latent Head)
# We take index [0] because LSTM layers return [output, hidden_state, cell_state]
hidden_states = visual_encoder.get_layer('lstm_pre_training').output[0]

# 3. Construct the new Multi-Output Model
# Keras will trace the path from these two outputs back to encoder_inputs
grafted_encoder = Model(
    inputs=encoder_inputs, 
    outputs=[predictions, hidden_states],
    name="grafted_visual_encoder"
)

grafted_encoder.summary()

Model: "grafted_visual_encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ground_truth_expr   │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 6, 15)     │          0 │ ground_truth_exp… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h0 (Dense)          │ (None, 256)       │     33,024 │ h_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c0 (Dense)          │ (None, 256)       │     33,024 │ c_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_pre_training   │ [(None, 6, 256),  │    278,528 │ scheduled_maskin… │
│ (LSTM)              │ (None, 256),      │            │ h0[0][0],         │
│                     │ (None, 256)]      │            │ c0[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_visual_enco… │ (None, 6, 15)     │      3,855 │ lstm_pre_trainin… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
def build_grafted_calculator(max_size=256, vocab_size=15):
    # 1. Inputs
    symbolic_expr = Input(shape=(6, vocab_size), name='expression_input')
    ans_teacher = Input(shape=(4, vocab_size), name='answer')
    encoder_latents = Input(shape=(6, max_size), name='encoder_latent_input')

    # 2. Hybrid Representation
    hybrid_encoder_repr = Concatenate(name="feature_fusion")([encoder_latents, symbolic_expr])

    # 3. Scheduled Masking
    masked_ans = Lambda(
    robust_scheduled_mask, 
    output_shape=(4, 15),
    name="scheduled_masking_layer"
    )(ans_teacher)

    # 4. Decoder LSTM
    decoder_outputs, _, _ = LSTM(max_size, return_sequences=True, return_state=True, name='decoder_lstm')(masked_ans)

    # 5. Global Attention
    # Note: This produces a context vector of depth 271
    attention_out = Attention(name='attention_block')([decoder_outputs, hybrid_encoder_repr])

    # 6. FEATURE PROJECTION (The Fix for 'combined_dense')
    # We project the 271-dim context back to 256-dim to match original weights
    projected_context = TimeDistributed(Dense(max_size), name='attention_projection')(attention_out)

    # 7. Concatenation (Shape now matches: 256 + 256 = 512)
    concat_q_A = Concatenate(name='concat_q_A')([decoder_outputs, projected_context])

    # 8. Combined Dense (Reasoning)
    combined_dense = Dense(max_size, name='combined_dense')(concat_q_A)

    # 9. Residual Path (The Fix for 'decoder_res_dense')
    # We MUST use use_bias=False to match the weight list length of 1
    decoder_res_dense = TimeDistributed(Dense(max_size, use_bias=False), name='decoder_res_dense')(ans_teacher)

    # 10. Residual Decoder Block
    decoder_with_residual = Add(name='decoder_with_residual')([combined_dense, decoder_res_dense])
    decoder_activation = TimeDistributed(Activation('relu'), name='decoder_activation')(decoder_with_residual)
    decoder_layer_norm = LayerNormalization(name='decoder_layer_norm')(decoder_activation)

    # 11. Output Head
    decoder_dense = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(decoder_layer_norm)

    return Model(inputs=[symbolic_expr, ans_teacher, encoder_latents], outputs=decoder_dense)

In [39]:
from tensorflow.keras.layers import Add, Activation, LayerNormalization
grafted_calculator = build_grafted_calculator()
grafted_calculator.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 4, 15)     │          0 │ answer[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_latent_inp… │ (None, 6, 256)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ scheduled_maskin… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_fusion      │ (None, 6, 271)    │          0 │ encoder_latent_i… │
│ (Concatenate)       │                   │            │ expression_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 271)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ feature_fusion[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_projecti… │ (None, 4, 256)    │     69,632 │ attention_block[… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_projec… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 487,695 (1.86 MB)

 Trainable params: 487,695 (1.86 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
def transfer_weights(source_model, target_model):
    """
    Transfers weights for critical reasoning layers identified in the summary.
    """
    critical_layers = [
        'decoder_lstm', 
        'combined_dense', 
        'decoder_res_dense', 
        'decoder_layer_norm', 
        'decoder_dense'
    ]
    
    print(f"{'Target Layer':<20} | {'Status'}")
    print("-" * 35)

    for name in critical_layers:
        try:
            src = source_model.get_layer(name)
            tgt = target_model.get_layer(name)
            
            src_weights = src.get_weights()
            tgt_weights = tgt.get_weights()
            
            # Dimension check to prevent crashes
            if all(s.shape == t.shape for s, t in zip(src_weights, tgt_weights)):
                tgt.set_weights(src_weights)
                print(f"{name:<20} transferred")
            else:
                print(f"{name:<20} not transferred")
        except Exception as e:
            print(f"{name:<20} else")

# Usage Implementation
grafted_calculator = build_grafted_calculator()
transfer_weights(calculator, grafted_calculator)

Target Layer         | Status
-----------------------------------
decoder_lstm         transferred
combined_dense       transferred
decoder_res_dense    transferred
decoder_layer_norm   transferred
decoder_dense        transferred


In [41]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Lambda

def build_full_e2e_pipeline(grafted_encoder, grafted_calculator):
    # 1. Define Pipeline Inputs (The 2 expected inputs)
    image_input = Input(shape=(5, 28, 28, 1), name="main_image_input")
    ans_teacher = Input(shape=(4, 15), name="main_answer_teacher")

    # 2. Modality Isolation (Create the zeros for the encoder)
    # We use a simple lambda that doesn't care about the 'training' flag
    expr_zeros = Lambda(
        lambda x: tf.zeros((tf.shape(x)[0], 6, 15), dtype=tf.float32),
        name="modality_isolation_zeros"
    )(image_input)

    # 3. Step 1: Encoder (Modality Isolation)
    # We set training=False to avoid signature errors in the encoder's internal Lambda
    enc_symbolic, enc_latent = grafted_encoder([image_input, expr_zeros], training=False)

    # 4. Step 2: Calculator (Reasoning + Attention)
    # The calculator STILL uses training=True (implicitly) for its own masking
    final_ans_output = grafted_calculator([enc_symbolic, ans_teacher, enc_latent])

    # 5. Build Final Model
    full_model = Model(
        inputs=[image_input, ans_teacher], 
        outputs=final_ans_output, 
        name="Hierarchical_E2E_Model"
    )
    
    return full_model

full_pipeline = build_full_e2e_pipeline(grafted_encoder, grafted_calculator)

In [33]:
full_pipeline.summary(expand_nested=True)

Model: "Hierarchical_E2E_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ main_image_input    │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ modality_isolation… │ (None, 6, 15)     │          0 │ main_image_input… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ grafted_visual_enc… │ [(None, 6, 15),   │  1,254,991 │ main_image_input… │
│ (Functional)        │ (None, 6, 256)]   │            │ modality_isolati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ sequence       │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ encoder_model  │ [(None, 5, 7, 7,  │    906,560 │ -                 │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ input_layer │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed    │ 1)                │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_1  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_2  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_3  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_4  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_5  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_7  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 14, 14, │        256 │ -               

 Total params: 1,742,686 (6.65 MB)

 Trainable params: 1,742,494 (6.65 MB)

 Non-trainable params: 192 (768.00 B)

In [43]:
# 1. Ensure your images are 5D (Batch, Time, Height, Width, Channels)
if len(X_train.shape) == 4:
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]

# 2. Define the training function with explicit X and Y
def train_warmup(model, X_img, Y_ans_in, Y_ans_target, val_data, epochs=5):
    # Freeze encoder for warm-up
    grafted_encoder.trainable = False
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    # Your custom callback
    sampling_callback = ScheduledSamplingCallback(sampling_prob, decay_rate=0.05, min_prob=0.7)
    
    print("Starting Warm-up Training...")
    model.fit(
        x=[X_img, Y_ans_in],         # EXPLICIT LIST OF 2 INPUTS
        y=Y_ans_target,              # EXPLICIT TARGET
        validation_data=( [val_data[0][0], val_data[0][1]], val_data[1] ),
        epochs=epochs,
        callbacks=[sampling_callback]
    )

# 3. Run the training
train_warmup(
    model=full_pipeline, 
    X_img=X_train, 
    Y_ans_in=y_train_in, 
    Y_ans_target=y_train_target,
    val_data=([X_val, y_val_in], y_val_target)
)

Starting Warm-up Training...
Epoch 1/5


NameError: Exception encountered when calling Lambda.call().

[1mname 'scheduled_mask_layer' is not defined[0m

Arguments received by Lambda.call():
  • inputs=tf.Tensor(shape=(None, 6, 15), dtype=float32)
  • mask=None
  • training=True

## Inference loop